In [0]:
%sql
SHOW CATALOGS;

catalog
adv_dev_westus3_demo_001
samples
system


In [0]:

SHOW SCHEMAS IN adv_dev_westus3_demo_001;


databaseName
default
information_schema


In [0]:
%sql

CREATE SCHEMA IF NOT EXISTS adv_dev_westus3_demo_001.ecommerce;


In [0]:
%sql
SHOW SCHEMAS IN adv_dev_westus3_demo_001;


databaseName
default
ecommerce
information_schema


In [0]:

%sql
CREATE VOLUME IF NOT EXISTS
adv_dev_westus3_demo_001.ecommerce.lakehouse_vol;


In [0]:

%sql
SHOW VOLUMES IN adv_dev_westus3_demo_001.ecommerce;


database,volume_name
ecommerce,lakehouse_vol


In [0]:

base_path = "/Volumes/adv_dev_westus3_demo_001/ecommerce/lakehouse_vol"

folders = [
    "Raw_DS",
    "bronze/orders",
    "bronze/order_items",
    "bronze/customers",
    "bronze/products",
    "bronze/payments",
    "silver",
    "gold"
]

for folder in folders:
    dbutils.fs.mkdirs(f"{base_path}/{folder}")

print("All project folders created successfully!")
/Volumes/adv_dev_westus3_demo_001/ecommerce/lakehouse_vol/Raw_DS/

All project folders created successfully!


In [0]:

base_path = "/Volumes/adv_dev_westus3_demo_001/ecommerce/lakehouse_vol"

display(dbutils.fs.ls(f"{base_path}/Raw_DS/"))



path,name,size,modificationTime
dbfs:/Volumes/adv_dev_westus3_demo_001/ecommerce/lakehouse_vol/Raw_DS/olist_customers_dataset.csv,olist_customers_dataset.csv,9033957,1792653835000
dbfs:/Volumes/adv_dev_westus3_demo_001/ecommerce/lakehouse_vol/Raw_DS/olist_order_items_dataset.csv,olist_order_items_dataset.csv,15438671,1792653851000
dbfs:/Volumes/adv_dev_westus3_demo_001/ecommerce/lakehouse_vol/Raw_DS/olist_order_payments_dataset.csv,olist_order_payments_dataset.csv,5777138,1792653824000
dbfs:/Volumes/adv_dev_westus3_demo_001/ecommerce/lakehouse_vol/Raw_DS/olist_orders_dataset.csv,olist_orders_dataset.csv,17654914,1792653853000
dbfs:/Volumes/adv_dev_westus3_demo_001/ecommerce/lakehouse_vol/Raw_DS/olist_products_dataset.csv,olist_products_dataset.csv,2379446,1792653805000
dbfs:/Volumes/adv_dev_westus3_demo_001/ecommerce/lakehouse_vol/Raw_DS/olist_sellers_dataset.csv,olist_sellers_dataset.csv,174703,1792653781000


In [0]:
# Base path
base_path = "/Volumes/adv_dev_westus3_demo_001/ecommerce/lakehouse_vol"

# Create parameter
dbutils.widgets.text("file_name", "ALL", "Enter file name or ALL")

file_name = dbutils.widgets.get("file_name").strip()

# Dataset configuration
datasets = {
    "orders": "olist_orders_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "products": "olist_products_dataset.csv",
    "payments": "olist_order_payments_dataset.csv"
}

# Select files dynamically
if file_name.upper() == "ALL":
    selected_datasets = datasets
else:
    selected_datasets = {
        folder: filename
        for folder, filename in datasets.items()
        if filename == file_name
    }

# Validate parameter
if not selected_datasets:
    raise ValueError(f"Invalid file name: {file_name}")

# Process selected files
for folder, filename in selected_datasets.items():

    source_path = f"{base_path}/Raw_DS/{filename}"
    target_path = f"{base_path}/bronze/{folder}"

    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(source_path)
    )

    row_count = df.count()

    df.write.mode("overwrite").parquet(target_path)

    print(f"{folder}: {row_count} rows written to Bronze")

print("Bronze ingestion completed successfully!")


orders: 99441 rows written to Bronze
order_items: 112650 rows written to Bronze
customers: 99441 rows written to Bronze
products: 32951 rows written to Bronze
payments: 103886 rows written to Bronze
Task 1 completed: All five datasets converted to Parquet.
